# D&D 5E Rules & Character Assistant

A minimal RAG baseline for the capstone, in the starter structure: a `search` function, a `build_prompt` function, an `llm` function, and a top-level `rag` function that chains them.

**Question this answers:** *D&D players need to look up rules mid-session, and generic LLM answers are often wrong and uncited. Can we answer rules/spell/character questions straight from the rulebooks and cite the book?*

**Data sources considered:**

1. **The official rulebook PDFs we already own** (in `data/pdfs/`) - chosen: works today, no API key or licensing question, and it is the exact content the app must cite.
2. The D&D 5e SRD (`dnd5eapi.co`, 5e.tools) - OGL-licensed and much easier to parse, but it only covers SRD content, not the full rulebooks.
3. Synthetic character sheets generated for the prototype - useful later for the "create a character" part of the app, skipped for this baseline.


In [1]:
import json
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from minsearch import Index
from pypdf import PdfReader

openai_client = OpenAI()

# Works whether Jupyter starts in the repo root or in notebooks/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA_DIR = ROOT / "data"
PDF_DIR = DATA_DIR / "pdfs"
CACHE = DATA_DIR / "documents.json"

print(f"project root: {ROOT}")
print(f"pdf files   : {len(list(PDF_DIR.glob('*.pdf')))}")

project root: /home/george/repos/dnd-rag-assistant
pdf files   : 10


## 1. Load Your Data

Each document is a dict with a few text fields to search over: `title`, `description`, `content`, `filename`.

Extracting text from 500MB of PDFs takes ~40s, so the result is cached in `data/documents.json` and reused on every later run.

In [2]:
DOCUMENTS_DIR = PDF_DIR

# Title and description for each PDF in the documents directory.
DOCUMENT_METADATA = [
    {
        "title": "Player's Handbook (D&D 5E)",
        "description": "Core rulebook for players: character creation, races, classes, "
        "backgrounds, equipment, spells and the basic rules of Dungeons & Dragons 5th Edition.",
        "filename": "D&D 5E - Player's Handbook.pdf",
    },
    {
        "title": "Dungeon Master's Guide (D&D 5E)",
        "description": "Core rulebook for the Dungeon Master: world building, running the game, "
        "adventure design, treasure and magic items for D&D 5th Edition.",
        "filename": "D&D 5E - Dungeon Master's Guide.pdf",
    },
    {
        "title": "Monster Manual (D&D 5E)",
        "description": "Compendium of monsters and creatures with stat blocks, lore and "
        "encounter guidelines for D&D 5th Edition.",
        "filename": "D&D 5E - Monster Manual.pdf",
    },
    {
        "title": "Eberron: Rising from the Last War (D&D 5E)",
        "description": "Campaign setting book for Eberron: the world after the Last War, "
        "the dragonmarked houses, Sharn, new races (warforged, changelings, shifters, kalashtar) "
        "and the artificer class.",
        "filename": "D&D 5E - Eberron - Rising from the Last War.pdf",
    },
    {
        "title": "Xanathar's Guide to Everything (D&D 5E)",
        "description": "Rules expansion with new subclasses, spells, downtime activities, "
        "tool proficiencies and DM tools such as encounter building and random tables.",
        "filename": "D&D 5E - Xanathar's Guide to Everything.pdf",
    },
    {
        "title": "Tasha's Cauldron of Everything (D&D 5E)",
        "description": "Rules expansion with new subclasses, optional class features, "
        "custom origin rules, spells, magic tattoos, sidekicks and puzzles.",
        "filename": "D&D 5E - Tasha's Cauldron of Everything.pdf",
    },
    {
        "title": "Volo's Guide to Monsters (D&D 5E)",
        "description": "Monster supplement with expanded lore on iconic creatures, new monster "
        "stat blocks and playable monstrous races.",
        "filename": "D&D 5E - Volo's Guide to Monsters.pdf",
    },
    {
        "title": "Mordenkainen's Tome of Foes (D&D 5E)",
        "description": "Monster supplement focused on the planes: demons, devils, elves, "
        "dwarves, halflings, gnomes and high-level threats with new stat blocks.",
        "filename": "D&D 5E - Mordenkainen's Tome of Foes.pdf",
    },
    {
        "title": "Diario di Campagna Eberron - Prologo (Giorni 1/7, Sessioni 1/3)",
        "description": "Appunti della campagna di Eberron: cronaca del prologo, giorni 1-7 "
        "(sessioni 1-3). La spedizione parte da Sharn, il dirottamento dell'aeronave verso "
        "una zona di manifestazione di Lamannia, l'incontro con Adalon e la Profezia draconica.",
        "filename": "5208a1d4-62b5-495b-8905-382e6f8f946d.pdf",
    },
    {
        "title": "Trascrizione Schede Personaggi D&D",
        "description": "Schede dei personaggi della campagna trascritte in formato strutturato "
        "ad alta leggibilita' per LLM: classe, livello, razza, caratteristiche e statistiche "
        "(es. Beatrishal Davian, Paladino 6 / Warlock 2 Kalashtar).",
        "filename": "Trascrizione Schede Personaggi D&D.pdf",
    },
]

In [3]:
def extract_pdf_text(pdf_path: Path) -> str:
    """Extract the full text content from a PDF file."""
    reader = PdfReader(str(pdf_path))
    return "\n".join(page.extract_text() or "" for page in reader.pages)


def load_documents():
    # Cached text: the notebook runs instantly on every run after the first.
    if CACHE.exists():
        print(f"loading cache {CACHE.relative_to(ROOT)}")
        return json.loads(CACHE.read_text())

    documents = []
    for meta in DOCUMENT_METADATA:
        pdf_path = DOCUMENTS_DIR / meta["filename"]
        documents.append(
            {
                "title": meta["title"],
                "description": meta["description"],
                "content": extract_pdf_text(pdf_path),
                "filename": meta["filename"],
            }
        )

    CACHE.write_text(json.dumps(documents))
    print(f"extracted {len(documents)} PDFs -> {CACHE.relative_to(ROOT)}")
    return documents


documents = load_documents()
print(f"Loaded {len(documents)} documents")

extracted 10 PDFs -> data/documents.json
Loaded 10 documents


In [4]:
# Preview: title, filename and content length of each document.
for doc in documents:
    print(f"Title:    {doc['title']}")
    print(f"Filename: {doc['filename']}")
    print(f"Chars:    {len(doc['content'])}")
    print("-" * 80)

Title:    Player's Handbook (D&D 5E)
Filename: D&D 5E - Player's Handbook.pdf
Chars:    1203595
--------------------------------------------------------------------------------
Title:    Dungeon Master's Guide (D&D 5E)
Filename: D&D 5E - Dungeon Master's Guide.pdf
Chars:    1190796
--------------------------------------------------------------------------------
Title:    Monster Manual (D&D 5E)
Filename: D&D 5E - Monster Manual.pdf
Chars:    1192376
--------------------------------------------------------------------------------
Title:    Eberron: Rising from the Last War (D&D 5E)
Filename: D&D 5E - Eberron - Rising from the Last War.pdf
Chars:    1350235
--------------------------------------------------------------------------------
Title:    Xanathar's Guide to Everything (D&D 5E)
Filename: D&D 5E - Xanathar's Guide to Everything.pdf
Chars:    731647
--------------------------------------------------------------------------------
Title:    Tasha's Cauldron of Everything (D&D 5E)
Fil

## 2. Chunk the Documents

Documents are huge - the biggest ones are over 1.2M characters. Sending 5 whole documents to the LLM would blow up the token count and drown the answer.

So we split each book into overlapping chunks with a sliding window (`gitsource.chunk_documents`, `size=3000`, `step=1500`), and index the chunks instead. The two small campaign-note PDFs are indexed whole.

In [5]:
from gitsource import chunk_documents

# Small enough to index as a single document.
EXCLUDE_FROM_CHUNKING = {
    "5208a1d4-62b5-495b-8905-382e6f8f946d.pdf",
    "Trascrizione Schede Personaggi D&D.pdf",
}

document_chunks = []
for doc in documents:
    if doc["filename"] in EXCLUDE_FROM_CHUNKING:
        # Whole-document chunk: keep all metadata, mark it as the only chunk.
        document_chunks.append({**doc, "start": 0, "chunk_id": 0})
    else:
        # chunk_documents preserves metadata and adds 'start' + chunked 'content'.
        for i, chunk in enumerate(chunk_documents([doc], size=3000, step=1500)):
            chunk["chunk_id"] = i
            document_chunks.append(chunk)

print(f"Created {len(document_chunks)} chunks from {len(documents)} documents")

Created 5517 chunks from 10 documents


In [6]:
from collections import Counter

chunks_per_doc = Counter(c["filename"] for c in document_chunks)
for filename, count in chunks_per_doc.most_common():
    print(f"{filename}: {count} chunks")

D&D 5E - Eberron - Rising from the Last War.pdf: 900 chunks
D&D 5E - Player's Handbook.pdf: 802 chunks
D&D 5E - Monster Manual.pdf: 794 chunks
D&D 5E - Dungeon Master's Guide.pdf: 793 chunks
D&D 5E - Mordenkainen's Tome of Foes.pdf: 686 chunks
D&D 5E - Volo's Guide to Monsters.pdf: 633 chunks
D&D 5E - Xanathar's Guide to Everything.pdf: 487 chunks
D&D 5E - Tasha's Cauldron of Everything.pdf: 420 chunks
5208a1d4-62b5-495b-8905-382e6f8f946d.pdf: 1 chunks
Trascrizione Schede Personaggi D&D.pdf: 1 chunks


## 3. Build the Index

`text_fields` are the text fields we search over, `keyword_fields` holds exact-match fields we can filter on (here: which book the chunk came from).

In [7]:
index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"],
)

index.fit(document_chunks)

print(f"Indexed {len(document_chunks)} chunks")

Indexed 5517 chunks


## 4. Define the RAG Functions

`search` retrieves relevant chunks and merges adjacent chunks from the same book into one passage, `build_prompt` formats them into a prompt, `llm` calls the model, and `rag` chains all three.

In [8]:
def merge_chunks(chunks):
    """Merge overlapping/adjacent chunks from the same document into one passage.

    Chunks know their absolute 'start' offset in the original document, so the
    overlap is written only once and the result is exact contiguous text.
    """
    chunks = sorted(chunks, key=lambda c: c["start"])
    merged = chunks[0]["content"]
    end = chunks[0]["start"] + len(merged)  # absolute end position in the document

    for chunk in chunks[1:]:
        chunk_end = chunk["start"] + len(chunk["content"])
        if chunk_end <= end:
            continue  # fully contained in what we already have
        overlap = max(end - chunk["start"], 0)
        if overlap == 0:
            merged += "\n...\n"  # gap between non-adjacent chunks
        merged += chunk["content"][overlap:]
        end = chunk_end

    return merged


BOOST = {"title": 3.0, "description": 2.0, "content": 1.0}


def search(query, num_results=3):
    """Search chunks, then merge adjacent chunks per book into passages."""
    # Fetch extra raw chunks: adjacent hits from the same book collapse
    # into a single merged passage.
    hits = index.search(query, num_results=num_results * 3, boost_dict=BOOST)

    grouped = {}
    for hit in hits:
        grouped.setdefault(hit["filename"], []).append(hit)

    passages = []
    for filename, chunks in grouped.items():
        content = merge_chunks(chunks) if len(chunks) > 1 else chunks[0]["content"]
        passages.append(
            {
                "title": chunks[0]["title"],
                "filename": filename,
                "content": content,
            }
        )

    return passages[:num_results]

In [9]:
instructions = """
You're a Dungeons & Dragons 5th Edition rules assistant. Your task is to answer
the QUESTION using only the provided CONTEXT, which contains excerpts from the
official D&D 5E rulebooks and the campaign's own notes.

- Cite the source (the "title" field) for every part of the answer, like
  (Player's Handbook, D&D 5E), so the player can check the book themselves.
- If the context doesn't contain the answer, say you don't know and name the
  book that would likely contain it.
- Answer in the same language the QUESTION is written in.
- For character-building questions, give the concrete rules/numbers (level
  prerequisites, spell slots, action economy) rather than general advice.
""".strip()

In [10]:
def build_prompt(query, search_results):
    search_result_json = json.dumps(search_results, indent=2)

    user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

    return user_prompt

In [11]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [12]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

## 5. Try It Out

A representative query: the kind of rules question a player actually asks mid-session.

In [13]:
query = "How does an opportunity attack work? What triggers it and what does it cost?"
print(rag(query))

An opportunity attack occurs when a creature moves out of your reach. It allows you to make a melee attack against that creature. Here are the specifics:

- **Trigger**: An opportunity attack is triggered when a creature you can see moves out of your reach (Player's Handbook, D&D 5E).
- **Cost**: Making an opportunity attack does not require an action, but it does count as your reaction for that turn (Player's Handbook, D&D 5E).

For more detailed information, you can refer to the Player's Handbook, which explains the mechanics of opportunity attacks thoroughly.


In [14]:
# And a character/spell question, to check the citation behaviour.
print(rag("Which spells can a 3rd-level wizard prepare, and where do the rules say that?"))

A 3rd-level wizard can prepare a number of spells based on their Intelligence modifier and their wizard level. Specifically, the rule states that a wizard can prepare a number of spells equal to their Intelligence modifier + their wizard level. 

For example, if a wizard has an Intelligence modifier of +2, they can prepare 5 spells (3 from their level + 2 from their modifier). The spells must be selected from the wizard spell list that they have access to (spells of levels for which they have spell slots).

For the exact rules, refer to the relevant definitions in the Player's Handbook:

- "Preparing Spells" section (Player's Handbook, D&D 5E).
- "Spell Slots" and "Spell List" sections (Player's Handbook, D&D 5E).

However, the context you provided does not include these specific rules directly. Therefore, for comprehensive details about which spells a 3rd-level wizard can prepare, I recommend checking the Player's Handbook, D&D 5E.
